# Clase 202 — Drift detection con PSI, K-S, Wasserstein y Evidently

Generamos un dataset de "producción" con shift sintético, detectamos con tests estadísticos manuales y con Evidently.

## 🧠 Intuición previa

Antes de las fórmulas (PSI, K-S, Wasserstein), quedate con el porqué:

> **El mundo cambia y el modelo envejece.** Entrenaste con la foto de un momento; la realidad se mueve. Monitorear la **distribución de los datos de entrada** es el termómetro que avisa *cuándo* el modelo se está quedando viejo y hay que reentrenar — antes de que la métrica de negocio se caiga.

Dos envejecimientos distintos, y por qué importan:

- **Data drift** (cambia P(X)): la distribución de las *entradas* se corre. Ejemplo: inflación hace que `ingreso` suba en todos lados; llega un segmento de clientes nuevo. El modelo no está "roto", pero opera en un terreno que no vio en el entrenamiento. **Se detecta sin labels** — solo mirando los inputs.
- **Concept drift** (cambia P(y|X)): la *relación* entre entrada y salida cambia. Ejemplo: post-pandemia, el mismo perfil de cliente ahora paga distinto. Aquí sí baja el desempeño real, y **solo se mide con labels** (o se *estima* con técnicas como CBPE).

La analogía: es como un mapa. El **data drift** es notar que estás caminando por calles que no figuran en tu mapa (entrada nueva). El **concept drift** es que las calles del mapa siguen ahí, pero ahora son de sentido contrario (la regla cambió). Los tests estadísticos de esta clase (PSI, K-S, Wasserstein) son formas de medir *cuánto* se corrió la distribución para disparar una alerta a tiempo.

## Setup

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
rng = np.random.default_rng(42)

# Reference = primer 60%, Current = último 40% con shift
n = len(X)
ref = X.iloc[:int(n * 0.6)].copy()
cur = X.iloc[int(n * 0.6):].copy()

# Shift sintético: 'MedInc' inflación + 'HouseAge' segmento nuevo
cur['MedInc'] = cur['MedInc'] * 1.5
cur = cur[cur['HouseAge'] >= 25]
print('ref:', ref.shape, '| cur:', cur.shape)

## 1. PSI manual

In [ ]:
def psi(ref, cur, bins=10):
    """Population Stability Index entre dos arrays continuos."""
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    p_ref, _ = np.histogram(ref, bins=edges)
    p_cur, _ = np.histogram(cur, bins=edges)
    p_ref = np.clip(p_ref / p_ref.sum(), 1e-6, 1)
    p_cur = np.clip(p_cur / p_cur.sum(), 1e-6, 1)
    return float(np.sum((p_cur - p_ref) * np.log(p_cur / p_ref)))

print(f'{"feature":15} {"PSI":>8}  interpretación')
for col in X.columns:
    p = psi(ref[col].values, cur[col].values)
    flag = '🟢 estable' if p < 0.1 else '🟡 cambio menor' if p < 0.2 else '🔴 SHIFT'
    print(f'{col:15} {p:>8.4f}  {flag}')

## 2. K-S y Wasserstein (continuas)

In [ ]:
results = []
for col in X.columns:
    ks = stats.ks_2samp(ref[col].values, cur[col].values)
    w = stats.wasserstein_distance(ref[col].values, cur[col].values)
    results.append({'feature': col, 'KS_stat': ks.statistic, 'KS_p': ks.pvalue, 'Wasserstein': w})
pd.DataFrame(results).round(4)

## 3. Sensibilidad de Wasserstein vs K-S a outliers

Si el shift es 'pocos puntos pero muy lejos', K-S puede no detectarlo y Wasserstein sí.

In [ ]:
a = rng.normal(0, 1, 5000)
b = a.copy()
outlier_idx = rng.choice(len(b), 50, replace=False)
b[outlier_idx] = b[outlier_idx] * 50   # 1% de outliers extremos

print('K-S:', stats.ks_2samp(a, b))
print('Wasserstein:', stats.wasserstein_distance(a, b))
print('PSI:', psi(a, b))
print('\n→ K-S puede no ver el cambio (centro intacto); Wasserstein y PSI sí.')

## 4. Reporte Evidently

In [ ]:
# Requiere: pip install evidently
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    rep = Report(metrics=[DataDriftPreset()])
    rep.run(reference_data=ref, current_data=cur)
    rep.save_html('drift_report.html')
    print('reporte guardado: drift_report.html')
    print('drift detectado en', sum(1 for m in rep.as_dict()['metrics'][0]['result']['drift_by_columns'].values() if m['drift_detected']), 'columnas')
except ImportError:
    print('pip install evidently para esta celda')

## 5. Alerta vía webhook (cooldown)

In [ ]:
import time, json, os
from pathlib import Path

def maybe_alert(webhook_url, message, cooldown_s=14400, marker_path=Path('.last_alert')):
    """Postea a webhook si no hubo alerta reciente. Stub seguro: no hace HTTP si webhook_url es None."""
    last = float(marker_path.read_text()) if marker_path.exists() else 0
    if time.time() - last < cooldown_s:
        print('[skipped] cooldown activo')
        return
    marker_path.write_text(str(time.time()))
    payload = {'text': message}
    if webhook_url:
        import httpx
        httpx.post(webhook_url, json=payload, timeout=5)
    print('[alert]', message)

drift_features = [r['feature'] for r in results if psi(ref[r['feature']], cur[r['feature']]) > 0.2]
if drift_features:
    maybe_alert(None, f'PSI > 0.2 en: {drift_features}. Reporte: s3://.../drift_report.html')
else:
    print('OK — sin drift accionable.')

## Ejercicio guiado

1. Tomá `prediction = model.predict(ref)` y `prediction_cur = model.predict(cur)`. Aplicá PSI sobre las distribuciones de predicción. ¿Coincide con el data drift?
2. Implementá CBPE (NannyML) para estimar `MAE` esperado en `cur` sin usar `y_cur`. Compará con el `MAE` real.
3. Hacé un dashboard Streamlit que toma snapshots diarios y plotea PSI por feature en los últimos 30 días.
4. Escribí un runbook con: (a) qué hacer si PSI dispara en `MedInc`, (b) qué hacer si dispara en TODAS las features (probable bug en pipeline upstream).

## Conclusiones

- PSI es el estándar simple e interpretable; Wasserstein capta lo que K-S no.
- Data drift ≠ concept drift — la única forma de medir el segundo es teniendo labels (o estimarlo con CBPE).
- Alerta sin cooldown + sin runbook = alert fatigue garantizada.
- Reference window fija hasta el siguiente retraining oficial.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. Casi todo es **ejecutable con la base científica** (numpy/scipy/sklearn). Evidently y NannyML no están instalados: para esos mostramos la **API real** y ejecutamos el *concepto* a mano (drift por columna con K-S; estimación de accuracy sin labels con confianza del modelo).

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
rng = np.random.default_rng(7)
n = len(X)
ref = X.iloc[:int(n * 0.6)].copy()
cur = X.iloc[int(n * 0.6):].copy()

def psi(ref, cur, bins=10):
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    p_ref, _ = np.histogram(ref, bins=edges)
    p_cur, _ = np.histogram(cur, bins=edges)
    p_ref = np.clip(p_ref / p_ref.sum(), 1e-6, 1)
    p_cur = np.clip(p_cur / p_cur.sum(), 1e-6, 1)
    return float(np.sum((p_cur - p_ref) * np.log(p_cur / p_ref)))
print('ref/cur listos; psi() definido.')

### Ejercicio 1 — PSI manual con deciles del reference

Bineamos `MedInc` en 10 deciles usando los **bordes del reference** (clave: los mismos bins para ambos). Verificamos PSI ≈ 0 sin shift y PSI grande con shift × 1.5.

In [ ]:
medinc_ref = ref['MedInc'].values
medinc_same = cur['MedInc'].values                 # sin shift
medinc_shift = cur['MedInc'].values * 1.5          # shift multiplicativo

psi_same = psi(medinc_ref, medinc_same)
psi_shift = psi(medinc_ref, medinc_shift)
print(f'PSI sin shift : {psi_same:.4f}  (esperado < 0.1)')
print(f'PSI x1.5 shift: {psi_shift:.4f}  (esperado > 0.25 => 🔴)')
assert psi_same < 0.1, 'sin shift el PSI debe ser chico'
assert psi_shift > 0.25, 'con shift x1.5 el PSI debe dispararse'
print('OK — PSI usa bordes del reference; mide cuánto se corrió la masa entre bins.')

### Ejercicio 2 — K-S vs Wasserstein ante outliers

Metemos outliers en 5% de producción (× 100). K-S mira la *máxima diferencia de CDF* (poco sensible a colas raras si el centro no se mueve); Wasserstein integra *cuánta masa se movió y cuán lejos* (sí lo ve). Reproducimos ambos casos.

In [ ]:
base = rng.normal(0, 1, 5000)
with_outliers = base.copy()
idx = rng.choice(len(with_outliers), int(0.05 * len(with_outliers)), replace=False)
with_outliers[idx] = with_outliers[idx] * 100     # 5% de outliers extremos

ks = stats.ks_2samp(base, with_outliers)
w = stats.wasserstein_distance(base, with_outliers)
print(f'K-S statistic : {ks.statistic:.4f}  (chico: el centro casi no cambió)')
print(f'Wasserstein   : {w:.4f}  (grande: masa lejana pesa)')
assert w > ks.statistic, 'Wasserstein capta el desplazamiento de masa que K-S subestima'
print('OK — para shifts de cola/outliers, Wasserstein es más sensible que K-S.')

### Ejercicio 3 — Reporte de drift (Evidently → concepto a mano)

Evidently corre un test por columna y marca `drift_detected`. Como no está instalado, replicamos el corazón: **K-S por columna** con umbral de p-value, tal como hace su `DataDriftPreset`.

In [ ]:
# API REAL:
#   from evidently.report import Report
#   from evidently.metric_preset import DataDriftPreset
#   rep = Report(metrics=[DataDriftPreset()]); rep.run(reference_data=ref, current_data=cur)
#   rep.save_html("drift_report.html")
cur_shift = cur.copy()
cur_shift['MedInc'] = cur_shift['MedInc'] * 1.5    # inducimos drift en una columna
def drift_by_columns(ref, cur, alpha=0.05):
    rows = []
    for col in ref.columns:
        p = stats.ks_2samp(ref[col].values, cur[col].values).pvalue
        rows.append({'feature': col, 'ks_pvalue': round(p, 4), 'drift_detected': p < alpha})
    return pd.DataFrame(rows)

report = drift_by_columns(ref, cur_shift)
print(report.to_string(index=False))
n_drift = int(report['drift_detected'].sum())
assert report.loc[report.feature == 'MedInc', 'drift_detected'].iloc[0]
print(f'\ncolumnas con drift: {n_drift} (MedInc entre ellas). Evidently haría este HTML por vos.')

### Ejercicio 4 — Concept drift sin labels (CBPE → concepto)

CBPE (NannyML) estima el desempeño en producción **sin labels**, usando la confianza del propio modelo (`predict_proba`). Lo replicamos: la accuracy esperada = promedio de la confianza en la clase predicha. Comparamos con la accuracy real (que sí tenemos en el ejercicio).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
# problema de clasificación: ¿casa cara? (y > mediana)
yc = (y > y.median()).astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, yc, test_size=0.4, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)

proba = clf.predict_proba(Xte)
conf = proba.max(axis=1)                    # confianza en la clase predicha
estimated_acc = float(conf.mean())          # CBPE: accuracy esperada sin ver y
actual_acc = accuracy_score(yte, clf.predict(Xte))
print(f'accuracy estimada (CBPE, sin labels): {estimated_acc:.3f}')
print(f'accuracy real     (con labels)      : {actual_acc:.3f}')
assert abs(estimated_acc - actual_acc) < 0.1, 'la estimación por confianza debe aproximar la real'
print('OK — CBPE alerta caídas de desempeño en prod aunque los labels lleguen tarde o nunca.')

### Ejercicio 5 — Alerta con webhook + cooldown

Script que (a) calcula PSI por feature, (b) postea a un webhook si alguna > 0.2, (c) respeta un cooldown de 4 h con un archivo marcador para no spamear. Ejecutable y seguro (no hace HTTP real).

In [ ]:
import time, json, tempfile
from pathlib import Path

def maybe_alert(features_drifted, webhook_url=None, cooldown_s=14400, marker=None):
    marker = marker or Path(tempfile.gettempdir()) / '.drift_last_alert'
    last = float(marker.read_text()) if marker.exists() else 0.0
    if time.time() - last < cooldown_s:
        return '[skipped] cooldown activo'
    marker.write_text(str(time.time()))
    payload = {'text': f'⚠️ Drift (PSI>0.2) en: {features_drifted}'}
    if webhook_url:
        import httpx; httpx.post(webhook_url, json=payload, timeout=5)   # solo si hay URL real
    return f'[alert] {json.dumps(payload)}'

cur_shift2 = cur.copy(); cur_shift2['MedInc'] *= 1.5
drifted = [c for c in X.columns if psi(ref[c].values, cur_shift2[c].values) > 0.2]
marker = Path(tempfile.mkdtemp()) / '.mark'
first = maybe_alert(drifted, marker=marker)     # dispara
second = maybe_alert(drifted, marker=marker)    # cooldown -> skip
print('1ª:', first)
print('2ª:', second)
assert first.startswith('[alert]') and second.startswith('[skipped]')
assert 'MedInc' in drifted
print('OK — alerta una vez y luego respeta el cooldown (anti alert-fatigue).')